# AI CV Personality Analyzer
## 02 - Data Preprocessing

### Project Overview

This notebook prepares the raw personality dataset for Natural Language
Processing (NLP) and Machine Learning.

The raw dataset contains CV/essay text and binary Big Five personality
labels:

- **O** — Openness
- **C** — Conscientiousness
- **E** — Extraversion
- **A** — Agreeableness
- **N** — Neuroticism

### Objectives

In this notebook, we will:

1. Load the raw train, validation, and test datasets.
2. Remove unnecessary index information.
3. Convert personality labels from strings to integers.
4. Validate the target labels.
5. Clean and normalize the text.
6. Handle empty text records.
7. Check duplicate records.
8. Verify the processed datasets.
9. Save the processed datasets for feature engineering.

### Important Data Leakage Rule

The predefined training, validation, and test splits will be preserved.

We will **not combine the three datasets** during preprocessing.

The raw data will remain unchanged.

## 1. Import Required Libraries

The following libraries are used for:

- Data manipulation
- Text processing
- Regular expressions
- File and directory management

In [24]:
# Import pandas for DataFrame operations
import pandas as pd

# Import NumPy for numerical operations
import numpy as np

# Import regular expressions for text cleaning
import re

# Import os for directory and file operations
import os

# Display all columns when viewing DataFrames
pd.set_option("display.max_columns", None)

# Display wider DataFrames in the notebook
pd.set_option("display.width", 120)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Define Dataset Paths

The raw datasets are stored in:

`data/raw/`

The processed datasets will be saved to:

`data/processed/`

The original raw files will not be modified.

In [25]:
# Define paths to the raw datasets
TRAIN_PATH = "../data/raw/train.parquet"
VALIDATION_PATH = "../data/raw/validation.parquet"
TEST_PATH = "../data/raw/test.parquet"

# Define the directory where processed datasets will be stored
PROCESSED_DIR = "../data/processed"

# Create the processed directory if it does not already exist
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Raw dataset paths configured.")
print("Processed data directory:", PROCESSED_DIR)

Raw dataset paths configured.
Processed data directory: ../data/processed


## 3. Load Raw Datasets

Load the three predefined dataset splits.

We keep the original train, validation, and test separation to avoid
data leakage during later Machine Learning stages.

In [26]:
# Load the training dataset
train_df = pd.read_parquet(TRAIN_PATH)

# Load the validation dataset
validation_df = pd.read_parquet(VALIDATION_PATH)

# Load the test dataset
test_df = pd.read_parquet(TEST_PATH)

# Display the dataset sizes
print("Datasets loaded successfully.")
print("-" * 40)
print(f"Training Set   : {train_df.shape}")
print(f"Validation Set : {validation_df.shape}")
print(f"Test Set       : {test_df.shape}")

Datasets loaded successfully.
----------------------------------------
Training Set   : (1578, 8)
Validation Set : (395, 8)
Test Set       : (494, 8)


## 4. Initial Data Validation

Before making any changes, verify that all expected columns are
available in each dataset.

Expected columns:

- O
- C
- E
- A
- N
- ptype
- text
- __index_level_0__

In [27]:
# Display columns in each dataset

print("Training columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(validation_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

Training columns:
['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__']

Validation columns:
['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__']

Test columns:
['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__']


## 5. Remove Unnecessary Index Column

The raw dataset contains:

`__index_level_0__`

This column represents the original row index from the source dataset.
It does not contain useful information for predicting personality.

We will remove it from all three datasets.

The original Parquet files remain unchanged.

In [28]:
# Name of the unnecessary source index column
INDEX_COLUMN = "__index_level_0__"

# Remove the column only if it exists
for df in [train_df, validation_df, test_df]:

    if INDEX_COLUMN in df.columns:
        df.drop(columns=[INDEX_COLUMN], inplace=True)

# Confirm the column has been removed
print("Index column removed.")

print("\nRemaining columns:")
print(train_df.columns.tolist())

Index column removed.

Remaining columns:
['O', 'C', 'E', 'A', 'N', 'ptype', 'text']


## 6. Define Personality Target Columns

The Big Five personality traits are the prediction targets:

- **O** — Openness
- **C** — Conscientiousness
- **E** — Extraversion
- **A** — Agreeableness
- **N** — Neuroticism

Each target is a binary classification problem with labels `0` and `1`.

In [29]:
# Define the Big Five personality target columns
TARGET_COLUMNS = ["O", "C", "E", "A", "N"]

# Verify that all target columns exist
missing_targets = [
    column for column in TARGET_COLUMNS
    if column not in train_df.columns
]

# Stop the pipeline if any target is missing
if missing_targets:
    raise ValueError(
        f"Missing personality target columns: {missing_targets}"
    )

print("All Big Five target columns are available.")
print(TARGET_COLUMNS)

All Big Five target columns are available.
['O', 'C', 'E', 'A', 'N']


## 7. Convert Personality Labels to Numeric Values

The raw dataset stores the Big Five labels as strings:

- `'0'`
- `'1'`

Machine Learning algorithms require numerical target values.

Therefore, we convert:

`'0' → 0`

`'1' → 1`

We apply the same conversion independently to the train,
validation, and test datasets.

In [30]:
# Convert each Big Five target from string labels to integers

for df in [train_df, validation_df, test_df]:

    for column in TARGET_COLUMNS:

        # Convert values to numeric
        df[column] = pd.to_numeric(
            df[column],
            errors="raise"
        ).astype(int)

print("Personality labels converted to integers.")

Personality labels converted to integers.


## 8. Validate Personality Labels

After conversion, each personality target should contain only:

- `0`
- `1`

Any other value indicates a data-quality problem.

In [31]:
# Validate the target values in all three datasets

for dataset_name, df in {
    "Train": train_df,
    "Validation": validation_df,
    "Test": test_df
}.items():

    print("\n" + "=" * 50)
    print(dataset_name)
    print("=" * 50)

    for column in TARGET_COLUMNS:

        unique_values = sorted(df[column].unique())

        print(f"{column}: {unique_values}")

        # Ensure only binary labels are present
        if not set(unique_values).issubset({0, 1}):
            raise ValueError(
                f"Unexpected labels found in {dataset_name} - {column}"
            )


Train
O: [0, 1]
C: [0, 1]
E: [0, 1]
A: [0, 1]
N: [0, 1]

Validation
O: [0, 1]
C: [0, 1]
E: [0, 1]
A: [0, 1]
N: [0, 1]

Test
O: [0, 1]
C: [0, 1]
E: [0, 1]
A: [0, 1]
N: [0, 1]


## 9. Analyze the Text Column

The `text` column contains the written content that will eventually
be used by the NLP model.

Before cleaning the text, we inspect:

- Missing values
- Empty strings
- Whitespace-only records
- Data type

In [32]:
# Check the text column before preprocessing

for dataset_name, df in {
    "Train": train_df,
    "Validation": validation_df,
    "Test": test_df
}.items():

    print("\n" + "=" * 50)
    print(dataset_name)
    print("=" * 50)

    print("Data type:", df["text"].dtype)
    print("Missing values:", df["text"].isnull().sum())

    # Convert temporarily to strings for empty-text checking
    text_values = df["text"].fillna("").astype(str)

    print(
        "Empty/whitespace-only records:",
        text_values.str.strip().eq("").sum()
    )


Train
Data type: object
Missing values: 0
Empty/whitespace-only records: 0

Validation
Data type: object
Missing values: 0
Empty/whitespace-only records: 0

Test
Data type: object
Missing values: 0
Empty/whitespace-only records: 0


## 10. Handle Missing Text

Text is the primary input for the personality prediction system.

Records without meaningful text cannot provide useful information
for NLP-based personality prediction.

We will:

1. Convert missing text to an empty string.
2. Remove records where the text contains only whitespace.

This operation is applied independently to each dataset split.

In [33]:
# Replace missing text values with empty strings
for df in [train_df, validation_df, test_df]:
    df["text"] = df["text"].fillna("").astype(str)

# Remove records with empty or whitespace-only text
train_df = train_df[
    train_df["text"].str.strip().ne("")
].copy()

validation_df = validation_df[
    validation_df["text"].str.strip().ne("")
].copy()

test_df = test_df[
    test_df["text"].str.strip().ne("")
].copy()

# Reset indexes after removing records
train_df.reset_index(drop=True, inplace=True)
validation_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

print("Empty text records removed.")

Empty text records removed.


## 11. Text Cleaning Function

The raw text may contain:

- HTML tags
- URLs
- Email addresses
- Extra whitespace
- Repeated spaces
- Unnecessary formatting

We create a reusable text-cleaning function.

### Important

We will **not aggressively remove words or punctuation** at this stage.

The goal is to clean obvious noise while preserving useful linguistic
information for NLP feature extraction.

In [34]:
def clean_text(text):
    """
    Clean raw text while preserving useful linguistic information.

    Processing steps:
    1. Convert input to string.
    2. Convert text to lowercase.
    3. Remove HTML tags.
    4. Replace URLs with a space.
    5. Replace email addresses with a space.
    6. Normalize repeated whitespace.
    7. Remove leading and trailing whitespace.
    """

    # Convert input to string
    text = str(text)

    # Convert text to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\b[\w\.-]+@[\w\.-]+\.\w+\b",
        " ",
        text
    )

    # Normalize repeated whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove leading and trailing whitespace
    text = text.strip()

    return text

## 12. Test the Text Cleaning Function

Before applying the cleaning function to thousands of records,
we test it on a small example.

In [35]:
# Example text for testing
sample_text = """
Hello!!! Visit https://example.com
or contact TEST@example.com.
This    contains   extra spaces.
"""

# Apply the cleaning function
cleaned_sample = clean_text(sample_text)

print("Original text:")
print(sample_text)

print("\nCleaned text:")
print(cleaned_sample)

Original text:

Hello!!! Visit https://example.com
or contact TEST@example.com.
This    contains   extra spaces.


Cleaned text:
hello!!! visit or contact . this contains extra spaces.


## 13. Apply Text Cleaning

Now we apply the same text-cleaning function to the train,
validation, and test datasets.

The cleaned text will remain in the same `text` column.

In [36]:
# Apply text cleaning to each dataset split

for df in [train_df, validation_df, test_df]:

    df["text"] = df["text"].apply(clean_text)

print("Text cleaning completed.")

Text cleaning completed.


## 14. Compare Text Before and After Cleaning

Let's inspect a few cleaned records to ensure that the preprocessing
did not remove useful information.

In [37]:
# Display a few cleaned text examples

display(
    train_df[["text"]].head(5)
)

,text
0,it is wednesday. i can't wait until friday bec...
1,"wow, i want to go talk to the socialist organi..."
2,"i wish polygamy was still legal. well, not pol..."
3,"well, lets see . . . i guess the foremost thin..."
4,college? i wonder how it will be? i just start...


## 15. Check Duplicate Records

Duplicate text records may cause the same document to appear more
than once.

We check duplicates based on the cleaned CV/text content.

This is important because duplicate records across train and test
sets could lead to overly optimistic model performance.

In [38]:
# Count duplicate text records within each dataset split

print("Duplicate text records")
print("-" * 40)

print(
    "Train:",
    train_df["text"].duplicated().sum()
)

print(
    "Validation:",
    validation_df["text"].duplicated().sum()
)

print(
    "Test:",
    test_df["text"].duplicated().sum()
)

Duplicate text records
----------------------------------------
Train: 0
Validation: 0
Test: 0


## 16. Cross-Split Duplicate Check

It is important to verify whether the same text appears in multiple
dataset splits.

If identical documents exist in both training and testing data,
the model could effectively see the test example during training.

This would be a form of data leakage.

In [39]:
# Convert text columns to sets for fast comparison

train_texts = set(train_df["text"])
validation_texts = set(validation_df["text"])
test_texts = set(test_df["text"])

# Check overlap between dataset splits

train_validation_overlap = train_texts.intersection(
    validation_texts
)

train_test_overlap = train_texts.intersection(
    test_texts
)

validation_test_overlap = validation_texts.intersection(
    test_texts
)

print("Cross-split duplicate analysis")
print("-" * 50)

print(
    "Train ↔ Validation:",
    len(train_validation_overlap)
)

print(
    "Train ↔ Test:",
    len(train_test_overlap)
)

print(
    "Validation ↔ Test:",
    len(validation_test_overlap)
)

Cross-split duplicate analysis
--------------------------------------------------
Train ↔ Validation: 0
Train ↔ Test: 0
Validation ↔ Test: 0


## 17. Handle Duplicate Records

Duplicate records within a dataset do not provide additional
information to the model.

We will remove duplicate text records **within each split**.

We will not move records between train, validation, or test sets.

The original dataset splits remain independent.

In [40]:
# Remove duplicate text records within each split

train_df = train_df.drop_duplicates(
    subset=["text"]
).reset_index(drop=True)

validation_df = validation_df.drop_duplicates(
    subset=["text"]
).reset_index(drop=True)

test_df = test_df.drop_duplicates(
    subset=["text"]
).reset_index(drop=True)

print("Duplicate text records removed within each split.")

Duplicate text records removed within each split.


## 18. Exclude `ptype` From Initial ML Features

The dataset contains a `ptype` column.

Because this column represents personality-related information,
using it as an input feature could introduce target leakage.

Therefore, `ptype` will not be used by the initial text-based
personality prediction model.

The column is removed from the processed datasets.

The Big Five targets remain unchanged.

In [41]:
# Remove the personality type column from all datasets

if "ptype" in train_df.columns:

    train_df.drop(columns=["ptype"], inplace=True)
    validation_df.drop(columns=["ptype"], inplace=True)
    test_df.drop(columns=["ptype"], inplace=True)

print("ptype removed from ML-ready datasets.")

ptype removed from ML-ready datasets.


## 19. Final Processed Dataset Structure

At this stage, each processed dataset should contain:

- `O`
- `C`
- `E`
- `A`
- `N`
- `text`

The Big Five columns are the target variables.

The `text` column is the primary input for future NLP feature engineering.

In [42]:
# Display the final columns in each dataset

print("Train columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(validation_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

Train columns:
['O', 'C', 'E', 'A', 'N', 'text']

Validation columns:
['O', 'C', 'E', 'A', 'N', 'text']

Test columns:
['O', 'C', 'E', 'A', 'N', 'text']


## 20. Final Data Validation

Perform final checks before saving the processed datasets.

We verify:

- Dataset shapes
- Missing values
- Target data types
- Target values
- Empty text

In [43]:
# Store processed datasets in a dictionary
processed_datasets = {
    "Train": train_df,
    "Validation": validation_df,
    "Test": test_df
}

# Perform final validation
for dataset_name, df in processed_datasets.items():

    print("\n" + "=" * 60)
    print(dataset_name)
    print("=" * 60)

    print("Shape:", df.shape)

    # Check missing values
    print(
        "Total missing values:",
        df.isnull().sum().sum()
    )

    # Check empty text
    print(
        "Empty text records:",
        df["text"].str.strip().eq("").sum()
    )

    # Check target data types
    print("\nTarget data types:")
    print(df[TARGET_COLUMNS].dtypes)

    # Check target values
    print("\nTarget unique values:")

    for column in TARGET_COLUMNS:
        print(
            f"{column}:",
            sorted(df[column].unique())
        )


Train
Shape: (1578, 6)
Total missing values: 0
Empty text records: 0

Target data types:
O    int32
C    int32
E    int32
A    int32
N    int32
dtype: object

Target unique values:
O: [0, 1]
C: [0, 1]
E: [0, 1]
A: [0, 1]
N: [0, 1]

Validation
Shape: (395, 6)
Total missing values: 0
Empty text records: 0

Target data types:
O    int32
C    int32
E    int32
A    int32
N    int32
dtype: object

Target unique values:
O: [0, 1]
C: [0, 1]
E: [0, 1]
A: [0, 1]
N: [0, 1]

Test
Shape: (494, 6)
Total missing values: 0
Empty text records: 0

Target data types:
O    int32
C    int32
E    int32
A    int32
N    int32
dtype: object

Target unique values:
O: [0, 1]
C: [0, 1]
E: [0, 1]
A: [0, 1]
N: [0, 1]


## 21. Final Dataset Summary

Create a compact summary of the processed datasets.

This allows us to verify the amount of data available for the next stage.

In [44]:
# Create a summary table for the processed datasets

summary_data = []

for dataset_name, df in processed_datasets.items():

    summary_data.append({
        "Dataset": dataset_name,
        "Records": len(df),
        "Columns": len(df.columns),
        "Missing Values": int(df.isnull().sum().sum()),
        "Empty Text": int(
            df["text"].str.strip().eq("").sum()
        )
    })

# Convert the summary into a DataFrame
processed_summary = pd.DataFrame(summary_data)

# Display the summary
display(processed_summary)

,Dataset,Records,Columns,Missing Values,Empty Text
0,Train,1578,6,0,0
1,Validation,395,6,0,0
2,Test,494,6,0,0


## 22. Save Processed Datasets

The cleaned datasets are now ready for the next stage.

They will be stored in:

`data/processed/`

Files:

- `train_processed.parquet`
- `validation_processed.parquet`
- `test_processed.parquet`

The raw files in `data/raw/` are not modified.

In [45]:
# Define output paths

TRAIN_OUTPUT = os.path.join(
    PROCESSED_DIR,
    "train_processed.parquet"
)

VALIDATION_OUTPUT = os.path.join(
    PROCESSED_DIR,
    "validation_processed.parquet"
)

TEST_OUTPUT = os.path.join(
    PROCESSED_DIR,
    "test_processed.parquet"
)

# Save processed datasets as Parquet files

train_df.to_parquet(
    TRAIN_OUTPUT,
    index=False
)

validation_df.to_parquet(
    VALIDATION_OUTPUT,
    index=False
)

test_df.to_parquet(
    TEST_OUTPUT,
    index=False
)

print("Processed datasets saved successfully.")
print("-" * 50)
print(TRAIN_OUTPUT)
print(VALIDATION_OUTPUT)
print(TEST_OUTPUT)

Processed datasets saved successfully.
--------------------------------------------------
../data/processed\train_processed.parquet
../data/processed\validation_processed.parquet
../data/processed\test_processed.parquet


## 23. Reload Saved Files

As a final verification step, reload the saved Parquet files.

This confirms that the files were written correctly and can be used
by future notebooks.

In [46]:
# Reload the saved processed datasets

train_check = pd.read_parquet(TRAIN_OUTPUT)
validation_check = pd.read_parquet(VALIDATION_OUTPUT)
test_check = pd.read_parquet(TEST_OUTPUT)

# Verify their shapes

print("Reload verification")
print("-" * 40)

print("Train:", train_check.shape)
print("Validation:", validation_check.shape)
print("Test:", test_check.shape)

Reload verification
----------------------------------------
Train: (1578, 6)
Validation: (395, 6)
Test: (494, 6)


## Conclusion

The data preprocessing stage is complete.

### Completed Steps

- Loaded the raw train, validation, and test datasets.
- Removed the unnecessary source index column.
- Converted Big Five labels from strings to integers.
- Validated binary personality targets.
- Handled missing and empty text records.
- Cleaned and normalized text.
- Checked duplicate records.
- Checked cross-split text overlap.
- Removed duplicate text within each split.
- Excluded the potentially leaking `ptype` column.
- Validated the final dataset structure.
- Saved ML-ready Parquet files.

### Processed Data

The processed datasets are stored in:

`data/processed/`

### Next Notebook

➡️ **03_feature_engineering.ipynb**

In the next stage, we will transform the cleaned CV text into numerical
features that Machine Learning algorithms can understand.

The feature engineering stage will focus on NLP techniques such as:

- TF-IDF
- Word and n-gram features
- Feature selection
- Personality-specific feature preparation